# NB08 — Enriched Signal Embedding (mem_exec + ata_read) + Conti_06 Comparison

**Motivation (HANDOFF update 11):** The current text embedding only uses `ata_write` + `mem_write`. Conti_06's delayed-encryption phase is behaviorally silent on writes, so the classifier calls it Benign. But Conti_06 has **63,549 ata_read events** (file enumeration) and **162 mem_exec events** (code loading) during that silent window — pre-encryption signals a delaying ransomware leaves behind.

**Change:** extend `log_lines_to_text()` to also tokenise `mem_exec` (code-execution page events) and `ata_read` (storage read events). Regenerate embeddings → retrain classifier → compare Conti_06 detection before/after.

**Token budget:** MiniLM max is 256 subtokens. Silent windows gain coverage; busy encryption windows may truncate slightly (already detected → acceptable).

**Outputs:**
- `features/text_sequences_enriched.npy` — (190, 300, 384) enriched embeddings
- `features/raw_text_windows_enriched.pkl` — raw enriched text windows
- `features/enriched_comparison_results.json` — before/after metrics

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedShuffleSplit
from sentence_transformers import SentenceTransformer
from pathlib import Path
import pickle, json, warnings
warnings.filterwarnings('ignore')

FEATURES_DIR = Path('features')
DATA_DIR     = Path('data')
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

T_WINDOW_NS = 100_000_000      # 0.1 s
T_D_NS      = 30_000_000_000   # 30 s
T_MAX_TOKENS = 200             # increased from 73 to allow 4 signal types

LABEL_3WAY = {
    'WannaCry':'Ransomware','Ryuk':'Ransomware','REvil':'Ransomware',
    'LockBit':'Ransomware','Darkside':'Ransomware','Conti':'Ransomware',
    'AESCrypt':'Legit-Encryption','SDelete':'Legit-Encryption','Zip':'Legit-Encryption',
    'Firefox':'Benign','Office':'Benign','Idle':'Benign',
}
VARIANT_CLASSES = [f'Conti_0{i}' for i in range(1, 8)]
CLASS_ORDER = ['Benign', 'Legit-Encryption', 'Ransomware']
print(f'Device: {device}')


/home/arshad/Claude/Projects/Directed Study/Ransomware/ransmap_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


## 1. Enriched tokenizer — 4 signal types

Added vs original:
- **`ata_read`**: `storage-read` + size label (small/medium/large). File enumeration phase — ransomware scans targets before encrypting.
- **`mem_exec`**: `mem-exec` + page type (4kb/2mb/mmio). Code-loading phase — DLL injection / payload decompression produces distinctive exec patterns.

Null tokens only for write signals (preserves original behaviour for write-silent windows). Read/exec tokens are additive — only present when events exist.

In [2]:
def size_label_write(size):
    return 'large-write' if size > 65536 else 'medium-write' if size > 4096 else 'small-write'

def size_label_read(size):
    return 'large-read' if size > 65536 else 'medium-read' if size > 4096 else 'small-read'

def entropy_label(entropy):
    if entropy < 0:    return 'unknown-entropy'
    if entropy > 0.9:  return 'very-high-entropy'
    if entropy > 0.7:  return 'high-entropy'
    if entropy > 0.4:  return 'moderate-entropy'
    if entropy > 0.1:  return 'low-entropy'
    return 'near-zero-entropy'

PAGE_TYPE = {1: '4kb-page', 2: '2mb-page', 4: 'mmio'}

def log_lines_to_text(ata_write_df, mem_write_df, mem_exec_df, ata_read_df):
    """
    Enriched tokeniser: ata_write + mem_write (original) + mem_exec + ata_read (new).
    Returns a single string of space-separated tokens, truncated to T_MAX_TOKENS words.
    """
    tokens = []

    # ── ata_write (write entropy — primary ransomware signal) ──────────────
    if ata_write_df.empty:
        tokens.append('no-storage-write')
    else:
        for _, row in ata_write_df.iterrows():
            size    = int(row.iloc[3]) if not pd.isna(row.iloc[3]) else 0
            entropy = float(row.iloc[4]) if not pd.isna(row.iloc[4]) else -1.0
            tokens.append(f'storage-write {size_label_write(size)} {entropy_label(entropy)}')

    # ── mem_write (memory write entropy) ──────────────────────────────────
    if mem_write_df.empty:
        tokens.append('no-memory-write')
    else:
        for _, row in mem_write_df.iterrows():
            entropy  = float(row.iloc[4]) if not pd.isna(row.iloc[4]) else 0.0
            mem_type = int(row.iloc[5])   if not pd.isna(row.iloc[5]) else 1
            tokens.append(f'memory-write {PAGE_TYPE.get(mem_type,"unknown-page")} {entropy_label(entropy)}')

    # ── ata_read (file enumeration — NEW) ──────────────────────────────────
    for _, row in ata_read_df.iterrows():
        size = int(row.iloc[3]) if not pd.isna(row.iloc[3]) else 0
        tokens.append(f'storage-read {size_label_read(size)}')

    # ── mem_exec (code loading — NEW) ──────────────────────────────────────
    for _, row in mem_exec_df.iterrows():
        mem_type = int(row.iloc[5]) if not pd.isna(row.iloc[5]) else 1
        tokens.append(f'mem-exec {PAGE_TYPE.get(mem_type,"unknown-page")}')

    # Join and truncate
    full_text = ' . '.join(tokens)
    words = full_text.split()
    if len(words) > T_MAX_TOKENS:
        words = words[:T_MAX_TOKENS]
    return ' '.join(words)

print('Enriched tokeniser defined.')
# Quick sanity check
import pandas as _pd
_empty = _pd.DataFrame()
_ex = log_lines_to_text(_empty, _empty, _empty, _empty)
print(f'Empty window token: "{_ex}"')


Enriched tokeniser defined.
Empty window token: "no-storage-write . no-memory-write"


## 2. Enriched window extractor — reads all 4 CSV types

In [3]:
def read_csv_safe(path):
    """Read a CSV file, return empty DataFrame on any failure."""
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    try:
        return pd.read_csv(path, header=None)
    except Exception:
        return pd.DataFrame()

def get_ref_ns(*dfs):
    """Minimum first timestamp across all provided DataFrames."""
    ref = None
    for df in dfs:
        if not df.empty:
            t = int(df.iloc[0, 0]) * 1_000_000_000 + int(df.iloc[0, 1])
            ref = t if ref is None else min(ref, t)
    return ref

def add_rel_ns(df, ref_ns):
    if df.empty:
        return df
    df = df.copy()
    df['rel_ns'] = (df.iloc[:, 0].astype(np.int64) * 1_000_000_000 +
                   df.iloc[:, 1].astype(np.int64)) - ref_ns
    return df

def window_slice(df, t_start, t_end):
    if df.empty or 'rel_ns' not in df.columns:
        return pd.DataFrame()
    mask = (df['rel_ns'] >= t_start) & (df['rel_ns'] < t_end)
    return df[mask]

def extract_text_windows_enriched(exec_dir):
    """Read ata_write, mem_write, mem_exec, ata_read and produce 300 window strings."""
    exec_dir = Path(exec_dir)
    ata_w  = read_csv_safe(exec_dir / 'ata_write.csv')
    mem_w  = read_csv_safe(exec_dir / 'mem_write.csv')
    mem_ex = read_csv_safe(exec_dir / 'mem_exec.csv')
    ata_r  = read_csv_safe(exec_dir / 'ata_read.csv')

    ref_ns = get_ref_ns(ata_w, mem_w, mem_ex, ata_r)
    if ref_ns is None:
        return ['no-storage-write no-memory-write'] * 300

    ata_w  = add_rel_ns(ata_w,  ref_ns)
    mem_w  = add_rel_ns(mem_w,  ref_ns)
    mem_ex = add_rel_ns(mem_ex, ref_ns)
    ata_r  = add_rel_ns(ata_r,  ref_ns)

    windows = []
    for t in range(0, T_D_NS, T_WINDOW_NS):
        t_end = t + T_WINDOW_NS
        windows.append(log_lines_to_text(
            window_slice(ata_w,  t, t_end),
            window_slice(mem_w,  t, t_end),
            window_slice(mem_ex, t, t_end),
            window_slice(ata_r,  t, t_end),
        ))
    return windows   # 300 strings

print('Enriched extractor defined.')
# Smoke-test on one Conti_06 execution
import subprocess, os
meta_df  = pd.read_csv(FEATURES_DIR / 'metadata.csv')
c06_rows = meta_df[meta_df['label'] == 'Conti_06']
if len(c06_rows):
    test_dir  = Path(c06_rows.iloc[0]['exec_dir'])
    test_wins = extract_text_windows_enriched(test_dir)
    # Print first non-null-only window
    for i, w in enumerate(test_wins):
        if 'storage-read' in w or 'mem-exec' in w:
            print(f'Conti_06 first enriched window (idx {i}): {w[:200]}')
            break
    print(f'Total windows: {len(test_wins)}')
    read_wins  = sum(1 for w in test_wins if 'storage-read' in w)
    exec_wins  = sum(1 for w in test_wins if 'mem-exec' in w)
    write_wins = sum(1 for w in test_wins if 'storage-write' in w)
    print(f'Windows with storage-read : {read_wins}/300')
    print(f'Windows with mem-exec     : {exec_wins}/300')
    print(f'Windows with storage-write: {write_wins}/300')


Enriched extractor defined.
Conti_06 first enriched window (idx 0): no-storage-write . memory-write 2mb-page moderate-entropy . memory-write 2mb-page high-entropy . memory-write 2mb-page low-entropy . memory-write 2mb-page high-entropy . memory-write 2mb-page very-hig
Total windows: 300
Windows with storage-read : 6/300
Windows with mem-exec     : 8/300
Windows with storage-write: 300/300


## 3. Generate enriched text windows + MiniLM embeddings

In [4]:
ENRICHED_CACHE = FEATURES_DIR / 'text_sequences_enriched.npy'
WINDOWS_CACHE  = FEATURES_DIR / 'raw_text_windows_enriched.pkl'

if ENRICHED_CACHE.exists():
    print('Loading cached enriched embeddings...')
    text_seqs_enr = np.load(ENRICHED_CACHE)              # (190, 300, 384)
    with open(WINDOWS_CACHE, 'rb') as f:
        all_windows_enr = pickle.load(f)
    labels  = meta_df['label'].values
    groups  = meta_df['group'].values
    labels_3way = np.array([LABEL_3WAY.get(l, l) for l in labels])
    print(f'Loaded: {text_seqs_enr.shape}')
else:
    print('Extracting enriched text windows...')
    labels, groups, all_windows_enr = [], [], []
    for i, row in meta_df.iterrows():
        windows = extract_text_windows_enriched(row['exec_dir'])
        all_windows_enr.append(windows)
        labels.append(row['label'])
        groups.append(row['group'])
        if (i+1) % 20 == 0:
            print(f'  {i+1}/{len(meta_df)} done')

    labels      = np.array(labels)
    groups      = np.array(groups)
    labels_3way = np.array([LABEL_3WAY.get(l, l) for l in labels])
    with open(WINDOWS_CACHE, 'wb') as f:
        pickle.dump(all_windows_enr, f)
    print(f'\nEmbedding with MiniLM...')

    embedder = SentenceTransformer('all-MiniLM-L6-v2')
    all_flat = [w for exec_wins in all_windows_enr for w in exec_wins]  # 190*300 strings
    batch_size = 512
    all_emb = []
    for start in range(0, len(all_flat), batch_size):
        batch = all_flat[start:start+batch_size]
        emb   = embedder.encode(batch, convert_to_numpy=True, show_progress_bar=False)
        all_emb.append(emb)
        if (start // batch_size) % 10 == 0:
            print(f'  {start}/{len(all_flat)} windows embedded')
    all_emb = np.concatenate(all_emb, axis=0)  # (57000, 384)
    text_seqs_enr = all_emb.reshape(len(meta_df), 300, 384).astype(np.float32)
    np.save(ENRICHED_CACHE, text_seqs_enr)
    print(f'\nSaved → {ENRICHED_CACHE}  shape: {text_seqs_enr.shape}')

orig_mask = groups == 'original'
var_mask  = groups == 'variant'
print(f'Ready. Originals: {orig_mask.sum()}  Variants: {var_mask.sum()}')


Extracting enriched text windows...
  20/190 done
  40/190 done
  60/190 done
  80/190 done
  100/190 done
  120/190 done
  140/190 done
  160/190 done
  180/190 done

Embedding with MiniLM...


Loading weights: 100%|██████████████████████| 103/103 [00:00<00:00, 4337.83it/s]


  0/57000 windows embedded
  5120/57000 windows embedded
  10240/57000 windows embedded
  15360/57000 windows embedded
  20480/57000 windows embedded
  25600/57000 windows embedded
  30720/57000 windows embedded
  35840/57000 windows embedded
  40960/57000 windows embedded
  46080/57000 windows embedded
  51200/57000 windows embedded
  56320/57000 windows embedded

Saved → features/text_sequences_enriched.npy  shape: (190, 300, 384)
Ready. Originals: 120  Variants: 70


## 4. Train 1D-CNN + measure Conti_06 detection — before vs after

In [6]:
class TextEmbeddingClassifier(nn.Module):
    def __init__(self, emb_dim=384, n_classes=3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(emb_dim, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 64,  kernel_size=5, padding=2),
            nn.BatchNorm1d(64),  nn.ReLU(),
        )
        self.classifier = nn.Sequential(
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, n_classes),
        )
    def forward(self, x):
        x_t    = x.permute(0, 2, 1)
        feat   = self.conv(x_t).mean(dim=-1)   # (batch, 64)
        return self.classifier(feat)

class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

def train_and_eval(seqs, labels, labels_3way, orig_mask, var_mask,
                   epochs=80, tag=''):
    """
    Train classifier on 80% of originals, evaluate on 20% holdout + all variants.
    Returns dict with metrics including per-variant Conti_06 recall.
    """
    le = LabelEncoder()
    le.fit(CLASS_ORDER)
    y_orig = le.transform(labels_3way[orig_mask])

    orig_seqs = seqs[orig_mask]    # (120, 300, 384)
    var_seqs  = seqs[var_mask]     # (70, 300, 384)
    var_labels = labels[var_mask]

    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    tr_idx, val_idx = next(sss.split(orig_seqs, y_orig))

    train_dl = DataLoader(SeqDataset(orig_seqs[tr_idx], y_orig[tr_idx]),
                          batch_size=16, shuffle=True)
    val_dl   = DataLoader(SeqDataset(orig_seqs[val_idx], y_orig[val_idx]),
                          batch_size=16, shuffle=False)

    torch.manual_seed(SEED)
    model = TextEmbeddingClassifier().to(device)
    opt   = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sch   = optim.lr_scheduler.StepLR(opt, step_size=25, gamma=0.5)
    crit  = nn.CrossEntropyLoss()

    for ep in range(epochs):
        model.train()
        for bx, by in train_dl:
            bx, by = bx.to(device), by.to(device)
            opt.zero_grad()
            crit(model(bx), by).backward()
            opt.step()
        sch.step()

    # Evaluate on val fold
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for bx, by in val_dl:
            bx, by = bx.to(device), by.to(device)
            correct += (model(bx).argmax(1) == by).sum().item()
            total   += len(by)
    val_acc = correct / total

    # Evaluate Conti variant detection
    var_dl = DataLoader(SeqDataset(var_seqs, np.zeros(len(var_seqs), dtype=int)),
                        batch_size=16, shuffle=False)
    var_preds = []
    with torch.no_grad():
        for bx, _ in var_dl:
            preds = model(bx.to(device)).argmax(1).cpu().numpy()
            var_preds.extend(preds)
    var_preds = np.array(var_preds)

    ransomware_cls = le.transform(['Ransomware'])[0]
    per_variant = {}
    for v in VARIANT_CLASSES:
        vm = var_labels == v
        if vm.sum() > 0:
            recall = (var_preds[vm] == ransomware_cls).mean()
            per_variant[v] = float(recall)

    overall_variant_recall = (var_preds == ransomware_cls).mean()

    print(f'[{tag}] val_acc={val_acc:.3f} | overall variant recall={overall_variant_recall:.3f}')
    for v, r in per_variant.items():
        flag = ' ← WATCH' if v == 'Conti_06' else ''
        print(f'  {v}: {r:.1%}{flag}')

    return dict(tag=tag, val_acc=val_acc,
                overall_variant_recall=overall_variant_recall,
                per_variant=per_variant)


In [7]:
# Load original embeddings (2-signal baseline)
text_seqs_orig = np.load(FEATURES_DIR / 'text_sequences.npy')  # (190, 300, 384)

print('=' * 55)
print('BASELINE — original 2-signal (ata_write + mem_write)')
print('=' * 55)
results_baseline = train_and_eval(
    text_seqs_orig, labels, labels_3way, orig_mask, var_mask,
    tag='baseline'
)

print()
print('=' * 55)
print('ENRICHED — 4-signal (+ mem_exec + ata_read)')
print('=' * 55)
results_enriched = train_and_eval(
    text_seqs_enr, labels, labels_3way, orig_mask, var_mask,
    tag='enriched'
)


BASELINE — original 2-signal (ata_write + mem_write)
[baseline] val_acc=0.958 | overall variant recall=0.886
  Conti_01: 100.0%
  Conti_02: 100.0%
  Conti_03: 100.0%
  Conti_04: 100.0%
  Conti_05: 100.0%
  Conti_06: 20.0% ← WATCH
  Conti_07: 100.0%

ENRICHED — 4-signal (+ mem_exec + ata_read)
[enriched] val_acc=0.917 | overall variant recall=0.886
  Conti_01: 100.0%
  Conti_02: 100.0%
  Conti_03: 100.0%
  Conti_04: 100.0%
  Conti_05: 100.0%
  Conti_06: 20.0% ← WATCH
  Conti_07: 100.0%


In [8]:
print('\n' + '=' * 65)
print('COMPARISON SUMMARY')
print('=' * 65)
print(f'{"Metric":<30} {"Baseline":>12} {"Enriched":>12} {"Delta":>8}')
print('-' * 65)

b, e = results_baseline, results_enriched
rows = [
    ('Val accuracy (3-way)', b['val_acc'], e['val_acc']),
    ('Overall variant recall', b['overall_variant_recall'], e['overall_variant_recall']),
]
for v in VARIANT_CLASSES:
    rows.append((f'  {v} recall', b['per_variant'].get(v,0), e['per_variant'].get(v,0)))

for label, bv, ev in rows:
    delta = ev - bv
    flag = ' ↑' if delta > 0.05 else (' ↓' if delta < -0.05 else '')
    watch = ' ← KEY' if 'Conti_06' in label else ''
    print(f'{label:<30} {bv:>12.3f} {ev:>12.3f} {delta:>+8.3f}{flag}{watch}')

# Save results
out = {'baseline': results_baseline, 'enriched': results_enriched}
with open(FEATURES_DIR / 'enriched_comparison_results.json', 'w') as f:
    json.dump(out, f, indent=2)
print(f'\nSaved → features/enriched_comparison_results.json')



COMPARISON SUMMARY
Metric                             Baseline     Enriched    Delta
-----------------------------------------------------------------
Val accuracy (3-way)                  0.958        0.917   -0.042
Overall variant recall                0.886        0.886   +0.000
  Conti_01 recall                     1.000        1.000   +0.000
  Conti_02 recall                     1.000        1.000   +0.000
  Conti_03 recall                     1.000        1.000   +0.000
  Conti_04 recall                     1.000        1.000   +0.000
  Conti_05 recall                     1.000        1.000   +0.000
  Conti_06 recall                     0.200        0.200   +0.000 ← KEY
  Conti_07 recall                     1.000        1.000   +0.000

Saved → features/enriched_comparison_results.json
